In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['SLURM_CPUS_PER_TASK'] = '256'
os.chdir("/cluster/home/hasv/project-files/teleconnections/d7.2-Use-case1")

current_dir = os.getcwd()
print("Current Directory:", current_dir)

In [ ]:
!hostname

In [ ]:
import numpy as np
import pandas as pd

# Settings
num_points = 2005 - 850
max_allowed_features = 4
logged_results = {}

def generate_data(forecast_horizon, init_lag, max_lag, num_irrelevant):
    # Set seed for reproducibility
    np.random.seed(42)

    num_irrelevants = 3 * num_irrelevant

    # Generate synthetic time series data for indices A, B, C, D
    time = np.arange(num_points)
    index_A = 10 * (np.sin(time / 10) + np.random.normal(0, 0.1, num_points))
    index_B = np.random.uniform(0, 1, num_points)
    index_C = 20 * (np.cos(time / 20) + np.random.normal(0, 0.1, num_points))
    index_D = np.random.uniform(-1, 1, num_points)

    # Create a DataFrame
    data = pd.DataFrame({
        'Time': 851 + time,
        'Index_A': index_A,
        'Index_B': index_B,
        'Index_C': index_C,
        'Index_D': index_D
    })

    # Generate irrelevant indices within the specified ranges
    for i in range(num_irrelevant):
        data[f'Index_F{i+1}'] = np.random.uniform(-1, 1, num_points)
    for i in range(num_irrelevant):
        data[f'Index_F{num_irrelevant+i+1}'] = np.random.uniform(0, 1, num_points)
    for i in range(num_irrelevant):
        data[f'Index_F{(2*num_irrelevant)+i+1}'] = np.random.uniform(0, 20, num_points)

    # Create lagged features for a range of lags (1 to max_lag)
    lagged_data = []
    for lag in range(init_lag, max_lag + 1):
        lagged_columns = {f'Lag_{lag}_{col}': data[col].shift(lag) for col in ['Index_A', 'Index_B', 'Index_C', 'Index_D']}
        for i in range(num_irrelevant):
            key = f'Index_F{i+1}'
            lagged_columns[f'Lag_{lag}_{key}'] = data[key].shift(lag)
        for i in range(num_irrelevant):
            key = f'Index_F{num_irrelevant+i+1}'
            lagged_columns[f'Lag_{lag}_{key}'] = data[key].shift(lag)
        for i in range(num_irrelevant):
            key = f'Index_F{(2*num_irrelevant)+i+1}'
            lagged_columns[f'Lag_{lag}_{key}'] = data[key].shift(lag)
        lagged_data.append(pd.DataFrame(lagged_columns))
    
    if(forecast_horizon > max_lag or forecast_horizon<init_lag ):
        lagged_columns = {f'Lag_{forecast_horizon}_{col}': data[col].shift(forecast_horizon) for col in ['Index_A', 'Index_B', 'Index_C', 'Index_D']}
        lagged_data.append(pd.DataFrame(lagged_columns))
        
    lagged_data = pd.concat(lagged_data, axis=1)
    data = pd.concat([data, lagged_data], axis=1)

    # Drop rows with NaN values (due to lagging)
    data = data.dropna().reset_index(drop=True)

    # Coefficients for generating Target_E
    coefficients = [0.5, -0.3, 0.2, 0.1]
    #print("data", data.columns)
        
    # Initialize the target variable with random noise
    Target_E = np.random.normal(0, 0.0001, len(data))

    # Add the contribution of each lagged index to the target variable
    for i, col in enumerate(['A', 'B', 'C', 'D']):
        Target_E += coefficients[i] * data[f'Lag_{forecast_horizon}_Index_{col}']
        
    if(forecast_horizon > max_lag or forecast_horizon<init_lag ):       
        for col in ['Index_A', 'Index_B', 'Index_C', 'Index_D']: del data[f'Lag_{forecast_horizon}_{col}'] 
        
    # Delete columns that don't have 'Lag' prefix
    Time = data["Time"].copy()
    data = data.loc[:, data.columns.str.startswith('Lag')].copy()
    data['Target_E'] = Target_E
    data["Time"] = Time
    data = data.sample(frac=1, axis=1)
    # Reorder columns
    cols = ['Time', 'Target_E'] + [col for col in data.columns if col not in ['Time', 'Target_E']]
    data = data[cols].copy()
    
    # Normalize the data from 0 to 1
    data = (data - data.min()) / (data.max() - data.min())
    data = data *100
    return data


In [ ]:
import matplotlib.pyplot as plt

def plot_mae_corr(num_irrelevant, logged_results, modelname="LinearRegression", filename="results.png"):
    

    logged_results_df = pd.DataFrame.from_dict(logged_results, orient='index')
    logged_results_df.to_csv( "results/dummy-results/"+modelname+"_"+filename[:-3] + 'csv', index=False)
    
    num_irrelevants = range(1, num_irrelevant)
    selected_features = [logged_results[f"{modelname}_{num}"]["selected_features"] for num in num_irrelevants]
    num_irrelevant_range = [logged_results[f"{modelname}_{num}"]["num_irrelevant"] for num in num_irrelevants]
    mae_scores = [logged_results[f"{modelname}_{num}"]["mae_score"] for num in num_irrelevants]
    corr_scores = [logged_results[f"{modelname}_{num}"]["corr_score"] for num in num_irrelevants]

    # Plot results
    fig, ax1 = plt.subplots(figsize=(15, 5))

    color = 'tab:blue'
    ax1.set_xticks(num_irrelevant_range)
    ax1.set_xticklabels(selected_features, rotation=90)
    ax1.set_xlabel('Number of Irrelevant Features')
    ax1.set_ylabel('Mean Absolute Error', color=color)
    ax1.plot(num_irrelevant_range, mae_scores, color=color, label='MAE')
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  # instantiate a second y-axis that shares the same x-axis

    color = 'tab:red'
    ax2.set_ylabel('Pearson Correlation Score', color=color)  # we already handled the x-label with ax1
    ax2.plot(num_irrelevant_range, corr_scores, color=color, label='Correlation')
    ax2.tick_params(axis='y', labelcolor=color)

    ax2.invert_yaxis()
    ax1.grid(True)
    ax2.grid(True)

    fig.tight_layout()  # to ensure the right y-label is not slightly clipped
    plt.title(f'MAE and Pearson Correlation vs Number of Irrelevant Features {modelname}')
    
    plt.savefig("results/dummy-results/"+modelname+"_"+filename)
    plt.show()

In [ ]:
for num_irrelevant in (range(1, 100)):
    forecast_horizon = 5
    max_lag=num_irrelevant
    init_lag=max(1, max_lag-5)
    
    #print(num_irrelevant, forecast_horizon > max_lag or forecast_horizon <init_lag )

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from tqdm import tqdm
import numpy as np
import pandas as pd
from IPython.display import clear_output


for num_irrelevant in tqdm(range(1, 50)):
    
    max_lag=num_irrelevant
    init_lag=max(1, max_lag-12)
    data = generate_data(forecast_horizon=15, init_lag=init_lag, max_lag=max_lag, num_irrelevant=20)

    # Prepare your data
    X = data.drop(['Time', 'Target_E'], axis=1)
    y = data['Target_E']

    # Split the data into train and test sets
    split_index = int(len(X) * 0.6)  # 60% for training, 40% for testing
    X_train, X_test = X[:split_index], X[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]

    n_ensembles = 1
    feature_importances = np.zeros(X_train.shape[1])

    # Train multiple Linear Regression models and aggregate feature importances
    for i in range(n_ensembles):
        model = LinearRegression()
        model.fit(X_train, y_train)
        feature_importances += np.abs(model.coef_)

    # Average the feature importances
    feature_importances /= n_ensembles

    selected_indices = np.argsort(feature_importances)[-max_allowed_features:]
    selected_features = X.columns[selected_indices]

    # Transform the data to keep only selected features
    X_train_selected = X_train.iloc[:, selected_indices]
    X_test_selected = X_test.iloc[:, selected_indices]

    # Retrain the model with selected features
    model_selected = LinearRegression()
    model_selected.fit(X_train_selected, y_train)

    # Predict and calculate MAE on the test set
    y_pred = model_selected.predict(X_test_selected)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Calculate Pearson correlation score
    corr_score, _ = pearsonr(y_test, y_pred)
    
    logged_results[f"LinearRegression_{num_irrelevant}"] = {
        "model": "LinearRegression",
        "num_irrelevant": num_irrelevant,
        "mae_score": mae,
        "corr_score": corr_score,
        "n_total_features": len(X.columns),
        "selected_features": ", ".join(selected_features.tolist()) + f"_{len(X.columns)}_{max_lag}"
    }
    # Clear notebook output
    # Display results
    
    #clear_output(wait=True)
    #print(selected_features.tolist())
plot_mae_corr(num_irrelevant, logged_results, modelname="LinearRegression", filename=str(num_points)+"_results.png")
    

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from deap import base, creator, tools, algorithms
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import random
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

def evaluate(individual, X_train, y_train):
    selected_features = [index for index, value in enumerate(individual) if value == 1]
    if len(selected_features) == 0:
        return 1e10, 0  # Penalize if no feature is selected
    X_train_selected = X_train[:, selected_features]
    model = LinearRegression()
    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_train_selected)
    mae = mean_absolute_error(y_train, y_pred)
    corr_score, _ = pearsonr(y_train, y_pred)
    return mae, corr_score

# Genetic Algorithm setup
def genetic_algorithm_feature_selection(X_train, y_train, num_generations=100, population_size=50):
    num_features = X_train.shape[1]
    
    # Only create classes if they don't already exist
    if not hasattr(creator, 'FitnessMulti'):
        creator.create("FitnessMulti", base.Fitness, weights=(-1.0, 1.0))  # Minimize MAE, Maximize Correlation
    if not hasattr(creator, 'Individual'):
        creator.create("Individual", list, fitness=creator.FitnessMulti)
    
    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=num_features)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    toolbox.register("evaluate", evaluate, X_train=X_train, y_train=y_train)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
    toolbox.register("select", tools.selNSGA2)
    
    population = toolbox.population(n=population_size)
    algorithms.eaMuPlusLambda(population, toolbox, mu=population_size, lambda_=population_size, cxpb=0.5, mutpb=0.2, ngen=num_generations, verbose=False)
    
    best_individual = tools.selBest(population, 1)[0]
    
    # Get indices of the top 6 features
    feature_importances = np.array(best_individual)
    top_6_features = np.argsort(feature_importances)[:6][::-1]
    
    return top_6_features

def process_iteration(num_irrelevant):
    max_lag = num_irrelevant
    init_lag = max(1, max_lag - 12)
    data = generate_data(forecast_horizon=15, init_lag=init_lag, max_lag=max_lag, num_irrelevant=20)

    # Prepare your data
    X = data.drop(['Time', 'Target_E'], axis=1)
    y = data['Target_E']

    # Split the data into train and test sets
    split_index = int(len(X) * 0.6)  # 60% for training, 40% for testing
    X_train, X_test = X[:split_index].values, X[split_index:].values
    y_train, y_test = y[:split_index].values, y[split_index:].values

    # Genetic Algorithm for feature selection
    best_individual = genetic_algorithm_feature_selection(X_train, y_train)
    selected_indices = best_individual
    selected_features = X.columns[selected_indices]

    # Transform the data to keep only selected features
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]

    # Retrain the model with selected features
    model_selected = LinearRegression()
    model_selected.fit(X_train_selected, y_train)

    # Predict and calculate MAE on the test set
    y_pred = model_selected.predict(X_test_selected)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Calculate Pearson correlation score
    corr_score, _ = pearsonr(y_test, y_pred)
    
    result = {
        "model": "GeneticAlgorithm",
        "num_irrelevant": num_irrelevant,
        "mae_score": mae,
        "corr_score": corr_score,
        "n_total_features": len(X.columns),
        "selected_features": ", ".join(selected_features.tolist()) + f"_{len(X.columns)}_{max_lag}"
    }
    
    return num_irrelevant, result

num_iterations = range(1, 50)
num_workers = min(len(num_iterations), 256)  # Use 256 cores if available

results = Parallel(n_jobs=num_workers)(delayed(process_iteration)(num_irrelevant) for num_irrelevant in tqdm(num_iterations))

logged_results = {f"GeneticAlgorithm_{num_irrelevant}": result for num_irrelevant, result in results}
plot_mae_corr(num_irrelevant, logged_results, modelname="GeneticAlgorithm", filename="results.png")


In [ ]:
len(logged_results), num_irrelevant
modelname = "GeneticAlgorithm"
num_irrelevants = range(1, num_irrelevant)
selected_features = [logged_results[f"{modelname}_{num}"]["selected_features"] for num in num_irrelevants]
num_irrelevant_range = [logged_results[f"{modelname}_{num}"]["num_irrelevant"] for num in num_irrelevants]
mae_scores = [logged_results[f"{modelname}_{num}"]["mae_score"] for num in num_irrelevants]
corr_scores = [logged_results[f"{modelname}_{num}"]["corr_score"] for num in num_irrelevants]

In [ ]:
selected_features

In [ ]:
modelname = "GeneticAlgorithm"
num_irrelevants = range(1, num_irrelevant)
selected_features = [logged_results[f"{modelname}_{num}"]["selected_features"] for num in num_irrelevants]
num_irrelevant_range = [logged_results[f"{modelname}_{num}"]["num_irrelevant"] for num in num_irrelevants]
mae_scores = [logged_results[f"{modelname}_{num}"]["mae_score"] for num in num_irrelevants]
corr_scores = [logged_results[f"{modelname}_{num}"]["corr_score"] for num in num_irrelevants]

# Plot results
fig, ax1 = plt.subplots(figsize=(15, 5))

color = 'tab:blue'
ax1.set_xticks(num_irrelevant_range)
ax1.set_xticklabels(selected_features, rotation=90)
ax1.set_xlabel('Number of Irrelevant Features')
ax1.set_ylabel('Mean Absolute Error', color=color)
ax1.plot(num_irrelevant_range, mae_scores, color=color, label='MAE')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second y-axis that shares the same x-axis

color = 'tab:red'
ax2.set_ylabel('Pearson Correlation Score', color=color)  # we already handled the x-label with ax1
ax2.plot(num_irrelevant_range, corr_scores, color=color, label='Correlation')
ax2.tick_params(axis='y', labelcolor=color)

ax2.invert_yaxis()
ax1.grid(True)
ax2.grid(True)

fig.tight_layout()  # to ensure the right y-label is not slightly clipped
plt.title(f'MAE and Pearson Correlation vs Number of Irrelevant Features {modelname}')

#plt.savefig("results/dummy-results/"+modelname+"_"+filename)
plt.show()



In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms

# Genetic algorithm settings
population_size = 200
generations = 50
cxpb, mutpb = 0.5, 0.2  # Crossover and mutation probabilities

def eval_lr(individual, X_train, y_train, X_test, y_test):
    # Select features based on the individual
    selected_features = [i for i, bit in enumerate(individual) if bit == 1]
    if not selected_features:
        return 10000.0,  # Return a large error if no features are selected
    
    X_train_selected = X_train[:, selected_features]
    X_test_selected = X_test[:, selected_features]

    # Build and train the model
    model = LinearRegression()
    model.fit(X_train_selected, y_train)

    # Predict and calculate MAE on the test set
    y_pred = model.predict(X_test_selected)
    mae = mean_absolute_error(y_test, y_pred)

    return mae,

def genetic_feature_selection(X_train, y_train, X_test, y_test):
    num_features = X_train.shape[1]
    if 'FitnessMin' not in creator.__dict__:
        creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
    if 'Individual' not in creator.__dict__:
        creator.create("Individual", list, fitness=creator.FitnessMin)
    
    toolbox = base.Toolbox()
    toolbox.register("attr_bool", np.random.randint, 2)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, num_features)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("evaluate", eval_lr, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

    pop = toolbox.population(n=population_size)
    algorithms.eaSimple(pop, toolbox, cxpb=cxpb, mutpb=mutpb, ngen=generations, verbose=False)
    
    best_individual = tools.selBest(pop, k=1)[0]
    selected_features = [i for i, bit in enumerate(best_individual) if bit == 1]

    return selected_features


for num_irrelevant in tqdm(range(1, 30)):
    data = generate_data(forecast_horizon=35, max_lag=50, num_irrelevant=num_irrelevant)

    # Prepare your data
    X = data.drop(['Time', 'Target_E'], axis=1)
    y = data['Target_E']

    # Split the data into train and test sets
    split_index = int(len(X) * 0.6)  # 60% for training, 40% for testing
    X_train, X_test = X[:split_index], X[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]

    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Use genetic algorithm for feature selection
    selected_indices = genetic_feature_selection(X_train, y_train, X_test, y_test)
    selected_features = data.columns[selected_indices].tolist()

    # Transform the data to keep only selected features
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]

    # Retrain the model with selected features
    model_selected = LinearRegression()
    model_selected.fit(X_train_selected, y_train)

    # Predict and calculate MAE on the test set
    y_pred = model_selected.predict(X_test_selected)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Calculate Pearson correlation score
    corr_score, _ = pearsonr(y_test, y_pred)
    
    logged_results[f"GA-LinearRegression_{num_irrelevant}"] = {
        "model": "GA-LinearRegression",
        "num_irrelevant": num_irrelevant,
        "mae_score": mae,
        "corr_score": corr_score,        
        "n_total_features": len(X.columns),
        "selected_features": selected_features
    }

# Display results
plot_mae_corr(29, logged_results, modelname="GA-LinearRegression")

In [ ]:
# Display results
plot_mae_corr(num_irrelevant, logged_results, modelname="LinearRegression")


In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import numpy as np

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1)
y = data['Target_E']

# Split the data into train and test sets
split_index = int(len(X) * 0.8)  # 80% for training, 20% for testing
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

n_ensembles = 10
feature_importances = np.zeros(X_train.shape[1])

# Train multiple CatBoost models and aggregate feature importances
for i in tqdm(range(n_ensembles)):
    model = CatBoostRegressor(iterations=10000, random_state=42 + i, task_type='GPU', verbose=0)
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
selected_indices = np.where(feature_importances >= threshold)[0]
selected_features = X.columns[selected_indices]

print("Selected features:", selected_features)

# Transform the data to keep only selected features
X_train_selected = X_train.iloc[:, selected_indices]
X_test_selected = X_test.iloc[:, selected_indices]

# Retrain the model with selected features
model_selected = CatBoostRegressor(iterations=1000, random_state=42, task_type='GPU', verbose=0)
model_selected.fit(X_train_selected, y_train)

# Predict and calculate MAE on the test set
y_pred = model_selected.predict(X_test_selected)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error on the test set:", mae)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import numpy as np

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1)
y = data['Target_E']

# Split the data into train and test sets
split_index = int(len(X) * 0.8)  # 80% for training, 20% for testing
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

n_ensembles = 10
feature_importances = np.zeros(X_train.shape[1])

# Train multiple Linear Regression models and aggregate feature importances
for i in tqdm(range(n_ensembles)):
    model = LinearRegression()
    model.fit(X_train, y_train)
    feature_importances += np.abs(model.coef_)

# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
selected_indices = np.where(feature_importances >= threshold)[0]
selected_features = X.columns[selected_indices]

print("Selected features:", selected_features)

# Transform the data to keep only selected features
X_train_selected = X_train.iloc[:, selected_indices]
X_test_selected = X_test.iloc[:, selected_indices]

# Retrain the model with selected features
model_selected = LinearRegression()
model_selected.fit(X_train_selected, y_train)

# Predict and calculate MAE on the test set
y_pred = model_selected.predict(X_test_selected)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error on the test set:", mae)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import numpy as np

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1)
y = data['Target_E']

# Split the data into train and test sets
split_index = int(len(X) * 0.8)  # 80% for training, 20% for testing
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

n_ensembles = 10
feature_importances = np.zeros(X_train.shape[1])

# Train multiple Random Forest models and aggregate feature importances
for i in tqdm(range(n_ensembles)):
    model = RandomForestRegressor(n_estimators=10000, random_state=42 + i, n_jobs=-1)
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
selected_indices = np.where(feature_importances >= threshold)[0]
selected_features = X.columns[selected_indices]
# Transform the data to keep only selected features
X_train_selected = X_train.iloc[:, selected_indices]
X_test_selected = X_test.iloc[:, selected_indices]
print("Selected features:", selected_features)

# Retrain the model with selected features
model_selected = RandomForestRegressor(n_estimators=10000, random_state=42, n_jobs=-1)
model_selected.fit(X_train_selected, y_train)

# Predict and calculate MAE on the test set
y_pred = model_selected.predict(X_test_selected)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error on the test set:", mae)


In [ ]:
from xgboost import XGBRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import numpy as np

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1)
y = data['Target_E']

# Split the data into train and test sets
split_index = int(len(X) * 0.8)  # 80% for training, 20% for testing
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

n_ensembles = 10
feature_importances = np.zeros(X_train.shape[1])

# Train multiple XGBoost models and aggregate feature importances
for i in tqdm(range(n_ensembles)):
    model = XGBRegressor(n_estimators=100000, random_state=42 + i, n_jobs=-1, tree_method='gpu_hist', predictor='gpu_predictor')
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
# Average the feature importances
feature_importances /= n_ensembles

# Select features based on mean importance across all models
threshold = np.mean(feature_importances)
selected_indices = np.where(feature_importances >= threshold)[0]
selected_features = X.columns[selected_indices]

print("Selected features:", selected_features)

# Transform the data to keep only selected features
X_train_selected = X_train.iloc[:, selected_indices]
X_test_selected = X_test.iloc[:, selected_indices]

# Retrain the model with selected features
model_selected = XGBRegressor(n_estimators=100000, random_state=42, n_jobs=-1, tree_method='gpu_hist', predictor='gpu_predictor')
model_selected.fit(X_train_selected, y_train)

# Predict and calculate MAE on the test set
y_pred = model_selected.predict(X_test_selected)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error on the test set:", mae)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1)
y = data['Target_E']

# Train the Random Forest Regressor
model = RandomForestRegressor(n_estimators=10000, random_state=42)
model.fit(X, y)

# Get feature importances
importances = model.feature_importances_

# Select features based on importance
selector = SelectFromModel(model, threshold='mean', prefit=True)
X_selected = selector.transform(X)

selected_features = X.columns[selector.get_support()]
print("Selected features:", selected_features)


In [ ]:
from pytorch_tabnet.callbacks import Callback

class VerboseCallback(Callback):
    def __init__(self, every_n_epochs=1000):
        self.every_n_epochs = every_n_epochs
    
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n_epochs == 0:
            print(f"Epoch {epoch + 1}: {logs}")


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from pytorch_tabnet.tab_model import TabNetRegressor
import numpy as np
import pandas as pd

# Prepare your data
X = data.drop(['Time', 'Target_E'], axis=1).values
y = data['Target_E'].values

# Normalize the features
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Define the number of splits for time series cross-validation
n_splits = 10

# Initialize the time series splitter
tscv = TimeSeriesSplit(n_splits=n_splits)

# Store results for each fold
results = []

class VerboseCallback:
    def __init__(self, every_n_epochs=1000):
        self.every_n_epochs = every_n_epochs
    
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n_epochs == 0:
            print(f"Epoch {epoch + 1}: {logs}")

verbose_callback = VerboseCallback(every_n_epochs=1000)

for fold, (train_index, test_index) in enumerate(tscv.split(X_normalized)):
    # Split the data into train and test sets using time series splitting
    X_train, X_test = X_normalized[train_index], X_normalized[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train TabNet with custom verbosity callback
    model = TabNetRegressor(verbose=0)
    model.fit(X_train=X_train, y_train=y_train.reshape(-1, 1), max_epochs=100000, callbacks=[verbose_callback])

    # Get feature importances
    importances = model.feature_importances_
    importance_indices = np.argsort(importances)[::-1]  # Sort indices of features by importance
    top_indices = importance_indices[:5]  # Get the top 5 features
    selected_features = data.columns[2:][top_indices]  # Get column names for top features
    
    print(f"Fold {fold + 1} - Selected features:", selected_features)

    # Predict on the test set
    y_test_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_test_pred)
    print(f"Fold {fold + 1} - Test MAE:", mae)
    
    # Store the result for the current fold
    results.append({
        'fold': fold + 1,
        'selected_features': selected_features.tolist(),
        'test_mae': mae
    })

# Convert results to DataFrame for better readability
results_df = pd.DataFrame(results)
print(results_df)

# Compute average MAE across folds
average_mae = np.mean(results_df['test_mae'])
print(f"Average MAE across all folds: {average_mae}")


In [ ]:
selected_features = data.columns[2:][importances > importances.mean()]
print("Selected features:", selected_features)

In [ ]:
importance_indices = np.argsort(importances)[::-1][:5]
selected_features = data.columns[2:][importance_indices]
print("Selected features:", selected_features)

In [ ]:

from deap import base, creator, tools, algorithms
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


# client = Client(cluster)


# Define the problem
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Custom initialization function to ensure individuals have at most 6 True values
def initIndividual():
    individual = [0] * ((len(data.columns) - 1))  # include target (also) as predictors
    selected_indices = np.random.choice(len(individual), max_allowed_features, replace=False)
    for idx in selected_indices:
        individual[idx] = 1
    return individual

toolbox.register("individual", tools.initIterate, creator.Individual, initIndividual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    # Ensure we do not select more than 6 features
    selected_indices = np.sum(individual)
    if selected_indices > max_allowed_features:
        return 1000.0,  # Penalize heavily if more than max_allowed_features features are selected

    # Select indices and lags based on individual
    selected_features = []
    for i, selected in enumerate(individual):
        if selected:
            feature_name = data.columns[i + 1]
            selected_features.append(feature_name)
    # If no features are selected, return a high error
    if not selected_features:
        return 1000.0,

    X = data[selected_features].copy().values     
    y = data['Target_E'].copy().values
    
    # Train linear regression model
    model = LinearRegression()
    model.fit(X, y)
    predictions = model.predict(X)
    mse = mean_squared_error(y, predictions)
    return mse,

# Custom mutation function to enforce the constraint
def mutate(individual, indpb):
    for i in range(len(individual)):
        if np.random.rand() < indpb:
            individual[i] = 1 - individual[i]
    
    # Ensure we do not have more than max_allowed_features True values
    selected_indices = np.sum(individual)
    if selected_indices > max_allowed_features:
        # If more than max_allowed_features are selected, reset excess to 0
        for _ in range(selected_indices - max_allowed_features):
            idx = np.random.choice(len(individual))
            if individual[idx] == 1:
                individual[idx] = 0
    return individual,

toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", mutate, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)

# Disable parallel processing by using the standard map function
toolbox.register("map", map)

def main():
    pop = toolbox.population(n=100)
    hof = tools.HallOfFame(1)

    # Statistics to monitor progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("std", np.std)

    for gen in range(10000):
        invalid_ind = [ind for ind in pop if not ind.fitness.valid]
        fitnesses = list(toolbox.map(toolbox.evaluate, invalid_ind))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Select the next generation individuals
        offspring = toolbox.select(pop, len(pop))
        # Clone the selected individuals
        offspring = list(map(toolbox.clone, offspring))

        # Apply crossover and mutation on the offspring
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if np.random.rand() < 0.5:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if np.random.rand() < 0.2:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = list(toolbox.map(toolbox.evaluate, invalid_ind))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Replace the old population by the offspring
        pop[:] = offspring

        # Update the hall of fame with the generated individuals
        hof.update(pop)

        # Append the current generation statistics to the logbook
        record = stats.compile(pop)
        print(record)

    return pop, stats, hof

pop, stats, hof = main()
print("Best individual is: ", hof[0])
selected_indices = []
for i, selected in enumerate(hof[0]):
    if selected:
        feature_name = data.columns[i + 1]
        selected_indices.append(feature_name)

print("Selected indices:", selected_indices)


In [ ]:
len(data.columns), ((len(data.columns) - 2) * max_lag), 2 + ((4+60)*(max_lag+1))